# Traffic Sign Detection: YOLOv8n on TT100K (50 Classes)

This notebook trains a compact real-time traffic sign detector for an automated driving capstone project.  
The final model is exported to ONNX for efficient CPU inference on Raspberry Pi.


## 1. Setup
- Install dependencies
- Import libraries
- Confirm TT100K dataset structure


In [ ]:
!pip -q install ultralytics pyyaml opencv-python

In [ ]:
import os
import yaml
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO

print('Libraries installed and imported successfully.')

In [ ]:
dataset_root = Path('/kaggle/input/datasets/braunge/tt100k/mydata')
assert dataset_root.exists(), f'Dataset not found: {dataset_root}'

required = [
    dataset_root / 'images' / 'train',
    dataset_root / 'images' / 'val',
    dataset_root / 'labels' / 'train',
    dataset_root / 'labels' / 'val',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing dataset paths:\n- ' + '\n- '.join(missing))

print('Dataset structure is valid.')

## 2. Build YOLO data config
Create a corrected YAML file in `/kaggle/working` so training uses an explicit class list.


In [ ]:
class_names = [
    'pl80', 'p6', 'ph', 'w', 'pa', 'p27', 'i5', 'p1', 'il70', 'p5',
    'pm', 'p19', 'ip', 'p11', 'p13', 'p26', 'i2', 'pn', 'p10', 'p23',
    'pbp', 'p3', 'p12', 'pne', 'i4', 'pb', 'pg', 'pr', 'pl5', 'pl10',
    'pl15', 'pl20', 'pl25', 'pl30', 'pl35', 'pl40', 'pl50', 'pl60',
    'pl65', 'pl70', 'pl90', 'pl100', 'pl110', 'pl120', 'il50', 'il60',
    'il80', 'il90', 'il100', 'il110'
]

data_config = {
    'path': str(dataset_root),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/val',
    'nc': len(class_names),
    'names': class_names,
}

yaml_path = Path('/kaggle/working/TT100K_corrected.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print('YAML saved to:', yaml_path)

## 3. Train YOLOv8n
For demo: 5 epochs. For production: 30-50 epochs.


In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project='tt100k_training',
    name='yolov8n_30e',
    exist_ok=True,
)

print('Training complete.')

## 4. Export ONNX
Export best checkpoint for edge deployment.


In [ ]:
best_model_path = Path('/kaggle/working/runs/detect/tt100k_training/yolov8n_30e/weights/best.pt')
model = YOLO(str(best_model_path))
model.export(format='onnx', imgsz=640)
print('ONNX export complete:', str(best_model_path).replace('.pt', '.onnx'))

## 5. Inference Preview
Run prediction on a few validation images and visualize output.


In [ ]:
val_images_dir = dataset_root / 'images' / 'val'
sample_images = list(val_images_dir.glob('*.jpg'))[:3]

for img_path in sample_images:
    preds = model(str(img_path))
    annotated = preds[0].plot()
    plt.figure(figsize=(9, 9))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f'Predictions on {img_path.name}')
    plt.axis('off')
    plt.show()

## 6. Credits
- TT100K dataset (Kaggle mirror: `braunge/tt100k`)
- Ultralytics YOLOv8
